# Computational Theory - Jamie Walsh

## Contents

- [Problem 1: Representing SHA-256 data](#problem-1-representing-sha-256-data)
- [Problem 2: SHA-256 Bitwise Operations](#problem-2-sha-256-bitwise-operations)
- [Problem 3: Generating the SHA-256 Constants](#problem-3-generating-the-sha-256-constants)
- [Problem 4: Padding and Parsing Messages](#problem-4-padding-and-parsing-messages)

In [23]:
import numpy as np
import sys
import struct

## Problem 1: Representing SHA-256 data

Before beginning any work on implementing SHA-256, this section will cover how we represent SHA-256's inputs, outputs and intermediate data in Python, and why these representations are appropriate.

### 32-bit words

Why 32 bits? SHA-256 requires 32-bit words, as specified in [FIPS 180-4 Section 3.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13). Within that detail there's a constraint which is important for how we represent this data: values are limited to 32 bits and overflow if this limited is exceeded.

Unlike "lower-level" languages like C++, where a 32-bit integer comes as a built-in `uint32_t`, Python's `int` does not behave this way.

We need to use a data type that won't exceed 32 bits. To test this, we can take the max value and add 1. Python's plain `int` has no clean way to enforce this limit; however, [NumPy's `np.iinfo`](https://stackoverflow.com/questions/23189506/maximum-allowed-value-for-a-numpy-data-type) provides a solution.

In [24]:
limit = np.iinfo(np.uint32).max

print(f"int: {limit + 1}")  # plain int: no limit, keeps growing
print(f"np.uint32: {np.uint32(limit) + 1}")  # np.uint32: wraps back to 0

int: 4294967296
np.uint32: 0


/var/folders/3q/zzb1pzp9655bb6h57vz921yh0000gn/T/ipykernel_8207/1439755418.py:4: RuntimeWarning: overflow encountered in scalar add
  print(f"np.uint32: {np.uint32(limit) + 1}")  # np.uint32: wraps back to 0


This confirms we need `np.uint32` as it wraps the 32-bit limit while Python's plain `int` does not, which FIPS 180-4 requires.

### Sequences of 32-bit words
Also specified in [FIPS 180-4 Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13), SHA-256 represents each 512-bit message block as a sequence of sixteen 32-bit words. To represent this, we use a NumPy array over a standard list, as [NumPy's documentation](https://numpy.org/doc/stable/user/whatisnumpy.html) states arrays are more efficient when every element is the same type, which applies here as every element is a `uint32`.

We can verify this claim in code:


In [25]:
numpy_array = np.array([1, 2, 3, 4, 5], dtype=np.uint32)
python_list = [np.uint32(x) for x in [1, 2, 3, 4, 5]]

print(f"Numpy Array Size: {numpy_array.nbytes}")  # array's total memory

# lists total memory: size of pointers + size of every object they point to
list_elements_size = sum(sys.getsizeof(x) for x in python_list)
list_size = sys.getsizeof(python_list) + list_elements_size
print(f"Python List Size: {list_size}")

print("numpy array:")
%timeit numpy_array + 1

print("python list:")
%timeit [x + 1 for x in python_list]

Numpy Array Size: 20
Python List Size: 260
numpy array:
507 ns ± 8.14 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
python list:
245 ns ± 2.75 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


From this data, the array (20 bytes) is much more efficient than the list (260 bytes) which supports NumPy's claims. However, we can see that the array was actually slower here (472ns vs 238ns) so why?

NumPy has a small, fixed overhead every time it performs an operation. With only 5 elements, that overhead outweighs any benefit. NumPy's speed advantage only shows up once arrays are large enough to outweigh this fixed cost.

So for our case, is this overhead actually an issue? Later problems use larger sequences, for example the 64-word message schedule in Problem 5. So we retest below at the same size, rather than assume this result generalises.


In [26]:
numpy_array = np.array(range(0, 64), dtype=np.uint32)
python_list = [np.uint32(x) for x in range(0, 64)]

print("numpy array:")
%timeit numpy_array + 1

print("python list:")
%timeit [x + 1 for x in python_list]

numpy array:
507 ns ± 9.39 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
python list:
2.49 μs ± 46.9 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


As predicted, retesting at a larger size (64 elements) shows the array is now significantly faster (486ns vs 2.44μs), roughly 5x faster. This confirms the array remains the right choice.

### Input Messages

Input messages can be stored as strings and converted to bytes using `.encode()` when required.

In [27]:
message = "abc"
message_bytes = message.encode()
print(message_bytes)

b'abc'


This confirms `.encode()` produces `bytes`, ready for further processing.

### 512-bit Message Blocks

A 512-bit message block is sixteen words. A block starts out as 64 bytes, so representing it as 16 words requires expressing those bytes as words. At a low level, these are all just bits, so rather than converting them (which costs computation), we can simply change how they're represented.

In [28]:
example_list = list(range(1, 17))

# '>16I': big-endian (>), 16 unsigned 32-bit ints (I)
packed_bytes = struct.pack('>16I', *example_list)
# struct.unpack: reads raw bytes back as a tuple of ints, no computation
unpacked = struct.unpack('>16I', packed_bytes)
print(f"Round-trip correct: {unpacked == tuple(example_list)}")

elements_size = sum(sys.getsizeof(x) for x in example_list)
list_size = sys.getsizeof(example_list) + elements_size
print(f"List size: {list_size}")
print(f"Packed bytes size: {sys.getsizeof(packed_bytes)}")

example_bytes = b'\x00\x00\x00\x05'

print("manual bit-shifting:")
%timeit (example_bytes[0] << 24) | (example_bytes[1] << 16) | (example_bytes[2] << 8) | example_bytes[3]

# '>I': one big-endian unsigned 32-bit int
print("struct.unpack:")
%timeit struct.unpack('>I', example_bytes)[0]

words_array = np.frombuffer(packed_bytes, dtype='>u4')
print(f"Words array: {words_array}")

Round-trip correct: True
List size: 632
Packed bytes size: 97
manual bit-shifting:
74.7 ns ± 0.907 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)
struct.unpack:
56.5 ns ± 0.481 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)
Words array: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]


The results confirm two things. First, the round trip check shows the conversion is lossless. Unpacking the packed bytes returns exactly the original sixteen numbers, so no information is lost by reinterpreting rather than computing. Second, the size and speed results show that reinterpreting bytes as words (type punning) is both more compact and faster than either storing the values in a list or manually reconstructing each word with bit-shifting

When reading through the problems, I remembered seeing clever time and cost saving measures with bits and bytes from the [Fast Inverse Square Root algorithm](https://down2core.com/docs/lomont.pdf). It uses the same underlying trick: reading the same bits through a different type, with no computation involved. The difference is what it's used for. Fast Inverse Square Root trades accuracy for speed, using this trick to get an approximate answer. Here, the same trick is used the opposite way, for an exact, lossless reformatting of data, since SHA-256 needs every value to be bit-exact

Also of note: Two format strings are used here `>16I` and `>u4`. Both start with `>` meaning big-endian, the reason this is important will be touched on more below.

### The final 256-bit hash value

[FIPS 180-4 Section 2.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=9) states that `H_0^(i)` "is the left-most word of hash value i". Since the hash value is 256 bits and a word is 32 bits, the hash value is exactly 8 words, numbered 0 to 7, left to right:

[0] [1] [2] [3] [4] [5] [6] [7]

In [29]:
hash_words = np.array(range(1, 9), dtype='>u4')  # 8 words matching [0] ... [7]
# words -> bytes, big-endian keeps word order matching [0] ... [7]
hash_bytes = hash_words.tobytes()
# bytes -> 64-character hex string, the final output format
hex_string = hash_bytes.hex()
print(hex_string)
print(len(hex_string))

0000000100000002000000030000000400000005000000060000000700000008
64


### Big-endian integers 

Back to the standard: [FIPS 180-4 Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=12) states that "Throughout this specification, the “big-endian” convention is used when expressing both 32- and 64-bit words, so that within each word, the most significant bit is stored in the left-most bit position".

So far we've been using `np.uint32` directly to represent a 32-bit word for arithmetic. Words, sequences, blocks and the hash value have all used it the same way without issue. This works everywhere except when converting to or from raw bytes. 

If we have a number `1234` and split it into `1`,`2`,`3`,`4`. We now have to decide which order to put them in, most significant first: (`1`,`2`,`3`,`4`) or least significant first: (`4`,`3`,`2`,`1`). The first is what "big-endian" means. The same applies to a `uint32`, converting it into 4 separate bytes splits it into pieces, and "which byte comes first" has to be consistent. A machine's native byte order may not answer the way FIPS 180-4 requires.

In [30]:
u32 = np.uint32  # native byte order -- safe for arithmetic
u32_be = '>u4'  # big-endian -- required when converting to/from raw bytes


def to_hex(word):
    """Write a 32-bit word as 8 hex digits, keeping leading zeros."""
    return format(word, '08x')


Going forward we will use two short aliases to make this distinction more explicit: `u32` for ordinary arithmetic, and `u32_be` specifically for the moments a word needs to become, or come from, raw bytes, where big-endian must be stated rather than assumed.

We will also often need to display a word in hex, so `to_hex` keeps this in one place. `format(word, '08x')` writes the word in hex (`x`) as 8 digits (`8`), filling any gaps with zeros (`0`).


### Summary of representation choices
- A **word** is `u32` (i.e. `np.uint32`), which enforces SHA-256's wraparound behavior
- A **sequence of words**, a **block** (16 words), and the **hash value** (8 words) are the same thing: a NumPy array of `u32` (i.e. `np.uint32`), with different lengths
- An **input message** is `bytes`, converted from a string with `.encode()`
- Bytes will be converted into words using **type punning** (`struct/np.frombuffer`) rather than manual computation
- **Big-endian** exists more as a rule for how conversions must be done rather than a type itself. `u32/u32_be` make this distinction explicit. 

## Problem 2: SHA-256 Bitwise Operations

SHA-256 will require some Boolean and bitwise functions for its actual implementation. These are seen in [FIPS 180-4 Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=15) as follows:
1. `Ch(x, y, z)`
2. `Maj(x, y, z)`
3. `Sigma0(x)` — written as Σ0{256}(x) in the standard
4. `Sigma1(x)` — written as Σ1{256}(x) in the standard
5. `sigma0(x)` — written as σ0{256}(x) in the standard
6. `sigma1(x)` — written as σ1{256}(x) in the standard

This section will cover each one's implementation, along with a brief explanation and test for each.

Since several of these functions take the same 3 boolean inputs (`x, y, z`), a shared testing setup is defined once here and reused for each, rather than repeating it per function. `run_truth_table` is generic and works for any number of inputs; `xyz` specifically enumerates all 8 combinations for a 3-input boolean function, reused by both `Ch` and `Maj` since they share that signature. Only the expected outputs differ per function.

In [31]:
xyz = [
    (0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1),
    (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1),
]

def run_truth_table(func, inputs, expected_values):
    for values, expected in zip(inputs, expected_values):
        result = func(*values)
        print(
            f"{func.__name__}{values} = {result}, "
            f"expected {expected}: {result == expected}"
        )

### Ch(x, y, z)

The Ch function: $Ch(x,y,z) = (x \land y) \oplus (\lnot x \land z)$

Broken down in pseudocode this becomes:
```pseudo
for each bit i:
    if x[i] == 1:
        result[i] = y[i]
    else:
        result[i] = z[i]
```

In [32]:
def Ch(x: u32, y: u32, z: u32) -> u32:
    """Ch function: For each bit positon, select y's bit if x is 1,
    else z's bit."""
    return (x & y) ^ (~x & z)

In [33]:
ch_expected = [0, 1, 0, 1, 0, 0, 1, 1]
run_truth_table(Ch, xyz, ch_expected)

Ch(0, 0, 0) = 0, expected 0: True
Ch(0, 0, 1) = 1, expected 1: True
Ch(0, 1, 0) = 0, expected 0: True
Ch(0, 1, 1) = 1, expected 1: True
Ch(1, 0, 0) = 0, expected 0: True
Ch(1, 0, 1) = 0, expected 0: True
Ch(1, 1, 0) = 1, expected 1: True
Ch(1, 1, 1) = 1, expected 1: True


All 8 truth table combinations match, confirming `Ch` correctly implements the choose function: selecting `y`'s bit where `x = 1`, and `z`'s bit where `x = 0`.

### Maj(x, y, z)

The Maj function: $Maj(x,y,z) =  (x \land y) \oplus (x \land z) \oplus ( y \land z)$

Broken down in pseudocode this becomes:
```pseudo
for each bit i:
    count = x[i] + y[i] + z[i]
    if count is >= to 2:
        result[i] = 1
    else:
        result[i] = 0
```

In [34]:
def Maj(x: u32, y: u32, z: u32) -> u32:
    """Majority vote per bit: 1 if at least 2 of x, y, z are 1, else 0."""
    return (x & y) ^ (x & z) ^ (y & z)

In [35]:
maj_expected = [0, 0, 0, 1, 0, 1, 1, 1]
run_truth_table(Maj, xyz, maj_expected)

Maj(0, 0, 0) = 0, expected 0: True
Maj(0, 0, 1) = 0, expected 0: True
Maj(0, 1, 0) = 0, expected 0: True
Maj(0, 1, 1) = 1, expected 1: True
Maj(1, 0, 0) = 0, expected 0: True
Maj(1, 0, 1) = 1, expected 1: True
Maj(1, 1, 0) = 1, expected 1: True
Maj(1, 1, 1) = 1, expected 1: True


All 8 truth table combinations match, confirming `Maj` correctly implements the majority vote: the result is 1 whenever at least 2 of `x, y, z` are 1.

### ROTR(x, n)

[FIPS 180-4 Section 3.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13) defines rotate right as `ROTR^n(x) = (x >> n) | (x << (32 - n))`: a normal right-shift combined with the same value shifted left, so bits that fall off the right appear at the top instead of being lost.

The implementation below is adapted from [this Stack Overflow thread](https://stackoverflow.com/questions/55856773/python-sha256-hash-computation), which shows a SHA-256 implementation producing incorrect output because its `ROTR` used plain Python `int`. Using `np.uint32` instead fixes this, since it enforces the 32-bit boundary automatically.

In [36]:
def ROTR(x: u32, n: int) -> u32:
    """Rotate x right by n bits."""
    return (x >> n) | (x << (32 - n))

### SHR(x, n)

[FIPS 180-4 Section 3.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13) defines shift right as `SHR^n(x) = x >> n`: bits shift right by `n` positions, any bits that fall off are discarded. Python has a shift right operator which we already use above (`>>`), which makes this function a very simple one-liner.

In [37]:
def SHR(x: u32, n: u32) -> u32:
    """Shift x right by n bits."""
    return x >> n

### Sigma0(x) and Sigma1(x)

[FIPS 180-4 Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=15) defines `Sigma0` and `Sigma1` with the same structure: three `ROTR` operations XORed together, differing only in the rotation amounts.

$Sigma0(x) = ROTR^2(x) \oplus ROTR^{13}(x) \oplus ROTR^{22}(x)$

$Sigma1(x) = ROTR^6(x) \oplus ROTR^{11}(x) \oplus ROTR^{25}(x)$

Sigma0's pseudocode:
```pseudo
a = rotate x right by 2
b = rotate x right by 13
c = rotate x right by 22
result = a XOR b XOR c
```

As Sigma0 and Sigma1 are identical with only their constants changing, we can abstract the function to reduce duplication.

In [38]:
def sigma_rotations(x, a, b, c):
    return ROTR(x, a) ^ ROTR(x, b) ^ ROTR(x, c)


def Sigma0(x: u32) -> u32:
    """Sigma0: ROTR by 2, 13, 22, XORed together."""
    return sigma_rotations(x, 2, 13, 22)


def Sigma1(x: u32) -> u32:
    """Sigma1: ROTR by 6, 11, 25, XORed together."""
    return sigma_rotations(x, 6, 11, 25)

### sigma0(x) and sigma1(x)

[FIPS 180-4 Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=15) defines `sigma0` and `sigma1` similarly to `Sigma0`/`Sigma1`, but the third operation `SHR` is shift right opposed to rotate right.

$sigma0(x) = ROTR^{7}(x) \oplus ROTR^{18}(x) \oplus SHR^{3}(x)$

$sigma1(x) = ROTR^{17}(x) \oplus ROTR^{19}(x) \oplus SHR^{10}(x)$

sigma0's pseudocode:
```pseudo
a = rotate x right by 7
b = rotate x right by 18
c = shift x right by 3
result = a XOR b XOR c
```

Same as before, we can do the same abstraction on the repeated pattern.

In [39]:
def sigma_shift(x, a, b, c):
    return ROTR(x, a) ^ ROTR(x, b) ^ SHR(x, c)

def sigma0(x: u32) -> u32:
    """sigma0: ROTR by 7, 18, then SHR by 3, XORed together."""
    return sigma_shift(x, 7, 18, 3)

def sigma1(x: u32) -> u32:
    """sigma1: ROTR by 17, 19, then SHR by 10, XORed together."""
    return sigma_shift(x, 17, 19, 10)

### Testing the Sigma and sigma functions

To assert the functions are correct we need real data for all four of the Sigma and sigma functions. Online I found this data hard to come by, except for [this SHA-256 visualiser](https://sha256algorithm.com/) ([source code](https://github.com/dmarman/sha256algorithm)). It hashes an input message step by step and shows the input and output of every function along the way.

Before trusting the tool we can check it. Hashing `abc` finishes with

`ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad`

which is the digest for `abc` given in [NIST's SHA-256 worked example](https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/SHA256.pdf), which the [Secure Hash Standard](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf) itself points to as it contains no worked examples. As the final hash is correct, we can assume the value at each step is too, and can therefore use them to test our functions.

From the visualiser, for the message `abc`:

| Function | Step on the tool | Input | Output |
|---|---|---|---|
| `sigma0` | "Calculate: w30" | `00000018` | `30060003` |
| `sigma0` | "Calculate: w31" | `61626380` | `940e90ef` |
| `sigma1` | "Calculate: w17" | `00000018` | `000f0000` |
| `sigma1` | "Calculate: w18" | `61626380` | `7da86405` |
| `Sigma0` | step 50 (round 0) | `6a09e667` | `ce20b47e` |
| `Sigma0` | step 51 (round 1) | `5d6aebcd` | `a29f64d1` |
| `Sigma1` | step 50 (round 0) | `510e527f` | `3587272b` |
| `Sigma1` | step 51 (round 1) | `fa2a4622` | `5a94fd2d` |

The test below runs each input through our function and compares the result to the tool's output. All eight rows should print `True`.

In [40]:
sigma_tests = [
    # (function, input, expected) -- values from sha256algorithm.com for "abc"
    (sigma0, 0x00000018, 0x30060003),
    (sigma0, 0x61626380, 0x940E90EF),
    (sigma1, 0x00000018, 0x000F0000),
    (sigma1, 0x61626380, 0x7DA86405),
    (Sigma0, 0x6A09E667, 0xCE20B47E),
    (Sigma0, 0x5D6AEBCD, 0xA29F64D1),
    (Sigma1, 0x510E527F, 0x3587272B),
    (Sigma1, 0xFA2A4622, 0x5A94FD2D),
]

for func, x, expected in sigma_tests:
    result = func(u32(x))
    print(f"{func.__name__}({to_hex(x)}) = {to_hex(result)}, "
          f"expected {to_hex(expected)}: {result == expected}")

assert all(func(u32(x)) == expected for func, x, expected in sigma_tests)

sigma0(00000018) = 30060003, expected 30060003: True
sigma0(61626380) = 940e90ef, expected 940e90ef: True
sigma1(00000018) = 000f0000, expected 000f0000: True
sigma1(61626380) = 7da86405, expected 7da86405: True
Sigma0(6a09e667) = ce20b47e, expected ce20b47e: True
Sigma0(5d6aebcd) = a29f64d1, expected a29f64d1: True
Sigma1(510e527f) = 3587272b, expected 3587272b: True
Sigma1(fa2a4622) = 5a94fd2d, expected 5a94fd2d: True


## Problem 3: Generating the SHA-256 Constants
[FIPS 180-4 Section 4.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16)
requires that the constants come from "the first sixty-four prime numbers", so we will need a function to return the first `n` primes.

The function below is adapted from [this Stack Overflow answer](https://stackoverflow.com/a/39106237), which compares five ways of generating primes. Each approach listed was tested with `n` set to 10,000 with the first four taking between roughly 0.004 and 1.1 seconds, while the fifth, took about 0.0005 seconds. (In the interest of not adding runtime and unnecessary code to the notebook, these tests can be seen in `roughwork/prime_timings.py`.)

There is some further discussion in the comments which affected the algorithm we've written:

1. The author later added an extra check, `sieve[p] % 2 == 1` described as filtering out odd values, which another commenter points out, does nothing, because `sieve[p]` is `True` or `False` and `True % 2` is always `1`. The author agrees the line could simply be `if sieve[p]`, but leaves the original code unchanged. We use the corrected line.
2. Another commenter states a speed up of roughly 24% by starting the list at `[2]` and only looping over odd numbers. For only 64 primes the speed makes no difference, so we've opted to keep the simpler loop.

The reference also answers a slightly different question. It returns every prime up to a limit `n`, whereas we need the first `n` prime numbers and do not know the limit, therefore includes some further slight changes. Our version starts with a small limit `z`, and if the sieve finishes without finding `n` primes, it doubles `z` and tries again. For `n = 64` this stops at `z = 512`, as the 64th prime is 311.


In [41]:
def primes(n: int) -> list:
    """Return the first n prime numbers."""
    z = 2
    while True:
        out = list()
        sieve = [True] * (z + 1)
        for p in range(2, z + 1):
            if sieve[p]:
                out.append(p)
                if len(out) == n:
                    return out
                for i in range(p, z + 1, p):
                    sieve[i] = False
        z *= 2


print(primes(64))



[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311]


### Cube Roots and Fractional Part of each cube root

Next we need to calculate the cube roots of the first 64 primes and extract the fractional part of each cube root: essentially, if our result was 1.5, we want the .5. NumPy supplies methods to do both of these.

In [42]:
def cube_root_fractions(primes_list: list) -> np.ndarray:
    """Return the fractional part of the cube root of each prime."""
    roots = np.cbrt(np.array(primes_list))
    return roots - np.floor(roots)

### Converting to 32-bit Unsigned Integer

Each fractional part is a number between 0 and 1, and we need its first 32 bits as a whole number. Multiplying by $2^{32}$ moves those bits in front of the decimal point, the same method used by [MMGen Wallet's SHA-2 implementation](https://mmgen.org/project/mmgen/mmgen-wallet/src/9638a96ec6782de93b3eddd4bbc4ca1a686d89b8/mmgen/sha2.py), and `np.floor` drops whatever is left.

To turn this into an integer we can use a type pun. Adding $2^{52}$ makes a 64-bit float store a whole number directly in its low bits, so we can read them straight out. [This Stack Overflow answer](https://stackoverflow.com/a/17035583) explains the trick. For the cube root of 2, the bits become `43300000428a2f98`, and the last 8 hex digits are the first constant. Without the floor this rounds instead of cutting off, and only 29 of the 64 constants match.

We read the bits the same way as Problem 1, using `.tobytes()` and `np.frombuffer`, but with `'>u8'` rather than `'>u4'` as a float is 8 bytes.

The aliases `f64_be` and `u64_be` are only needed here, as these values are not SHA-256 words. Everything else stays `u32`, including the constants themselves.


In [43]:
f64_be = '>f8'  # big-endian 64-bit float
u64_be = '>u8'  # big-endian 64-bit unsigned int

frac = cube_root_fractions(primes(64))
scaled = np.floor(frac * 2**32)  # first 32 fraction bits, as a whole number
root_bytes = (scaled + 2**52).astype(f64_be).tobytes()  # 2**52 fixes exponent
float_bits = np.frombuffer(root_bytes, dtype=u64_be)  # same bytes, as ints
K = float_bits.astype(u32)  # keep the low 32 bits
print(to_hex(K[0]))


428a2f98


### Displaying the Constants in Hexadecimal

Each constant is written as 8 hex digits, padded with zeros so that a value like `06ca6351` keeps its leading zero. They are printed in rows of 8 to match the layout used in the standard.

In [44]:
K_hex = [to_hex(k) for k in K]

for i in range(0, 64, 8):
    print(" ".join(K_hex[i:i + 8]))


428a2f98 71374491 b5c0fbcf e9b5dba5 3956c25b 59f111f1 923f82a4 ab1c5ed5
d807aa98 12835b01 243185be 550c7dc3 72be5d74 80deb1fe 9bdc06a7 c19bf174
e49b69c1 efbe4786 0fc19dc6 240ca1cc 2de92c6f 4a7484aa 5cb0a9dc 76f988da
983e5152 a831c66d b00327c8 bf597fc7 c6e00bf3 d5a79147 06ca6351 14292967
27b70a85 2e1b2138 4d2c6dfc 53380d13 650a7354 766a0abb 81c2c92e 92722c85
a2bfe8a1 a81a664b c24b8b70 c76c51a3 d192e819 d6990624 f40e3585 106aa070
19a4c116 1e376c08 2748774c 34b0bcb5 391c0cb3 4ed8aa4a 5b9cca4f 682e6ff3
748f82ee 78a5636f 84c87814 8cc70208 90befffa a4506ceb bef9a3f7 c67178f2


### Comparing with the Secure Hash Standard

[FIPS 180-4 Section 4.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16) lists all 64 constants, so we can copy them in and check our results against them directly.


In [45]:
fips_k = """
428a2f98 71374491 b5c0fbcf e9b5dba5 3956c25b 59f111f1 923f82a4 ab1c5ed5
d807aa98 12835b01 243185be 550c7dc3 72be5d74 80deb1fe 9bdc06a7 c19bf174
e49b69c1 efbe4786 0fc19dc6 240ca1cc 2de92c6f 4a7484aa 5cb0a9dc 76f988da
983e5152 a831c66d b00327c8 bf597fc7 c6e00bf3 d5a79147 06ca6351 14292967
27b70a85 2e1b2138 4d2c6dfc 53380d13 650a7354 766a0abb 81c2c92e 92722c85
a2bfe8a1 a81a664b c24b8b70 c76c51a3 d192e819 d6990624 f40e3585 106aa070
19a4c116 1e376c08 2748774c 34b0bcb5 391c0cb3 4ed8aa4a 5b9cca4f 682e6ff3
748f82ee 78a5636f 84c87814 8cc70208 90befffa a4506ceb bef9a3f7 c67178f2
""".split()

print(f"All 64 match FIPS 180-4: {K_hex == fips_k}")


All 64 match FIPS 180-4: True


### Recap: aliases and data structures

Before moving to problem 4, below is a minor recap of areas that might be cause confusion in what we've done so far 

| Name | What it is | Used for |
|---|---|---|
| `u32` | `np.uint32` | A word, for arithmetic |
| `u32_be` | `'>u4'` | A word when converting to or from bytes |
| `u64_be`, `f64_be` | `'>u8'`, `'>f8'` | Reading a float's bits in Problem 3 only, not SHA-256 words |
| Sequence of words | NumPy array of `u32` | A block (16 words), the hash value (8 words) and `K` (64 words) |
| Input message | `bytes` | The data being hashed |
| `K_hex`, `fips_k` | Lists of hex strings | Displaying and comparing `K` only |

Note that `Sigma0`/`Sigma1` and `sigma0`/`sigma1` are different functions: the capitalised pair uses three rotations, while the lowercase pair uses two rotations and a shift.


## Problem 4: Padding and Parsing Messages

SHA-256 works on 512-bit blocks, but a message can be any length. So before hashing, we pad the message to a multiple of 512 bits and split it into blocks.

Back to the standard: [FIPS 180-4 Section 5.1.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=18) says that for a message of $\ell$ bits we:

1. append a single `1` bit,
2. append $k$ zero bits, where $k$ is the smallest number (0 or more) such that $\ell + 1 + k \equiv 448 \pmod{512}$,
3. append $\ell$ as a 64-bit number.

[Section 5.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19) then splits the result into 512-bit blocks, each read as sixteen 32-bit words.

The equation in step 2 is the hard part. The [SHA-2 pseudocode on Wikipedia](https://en.wikipedia.org/wiki/SHA-2#Pseudocode) puts it more simply: keep adding zeros until the message, the `1` and the 64-bit length together make a multiple of 512.

```pseudo
begin with the original message of length L bits
append a single '1' bit
append K '0' bits, where K is the minimum number >= 0 such that (L + 1 + K + 64) is a multiple of 512
append L as a 64-bit big-endian integer, making the total post-processed length a multiple of 512 bits
```

Both say the same thing. The standard pads to 448 rather than 512 because it leaves room for the 64-bit length, and 448 + 64 = 512. Using the standard's own example, `abc` is 24 bits, so it gets one `1` bit, 448 - 25 = 423 zero bits and the 64-bit length: exactly one 512-bit block. The pseudocode also says the length is big-endian, which the standard covers in Section 3.1, as seen in Problem 1.

### Why pad this way?

The standard says how to pad, but not why. [This Reddit comment](https://www.reddit.com/r/cryptography/comments/tbj03i/comment/i081fpr/) explains it more simply:

- SHA-256 only works on whole blocks, so every input has to become blocks first.
- Two different messages must never become the same blocks, or they would give the same hash. So we must always be able to tell where the message ends and the padding begins.
- A `1` followed by zeros does this: take the length off the end, strip the zeros and the `1`, and the original message is left.
- The length is included because SHA-256's design, the [Merkle–Damgård construction](https://en.wikipedia.org/wiki/Merkle%E2%80%93Damg%C3%A5rd_construction), needs it for its security proof. It goes at the end because that's simpler to implement.
- Padding always adds at least 65 bits (the `1` and the 64-bit length). So if the last block has fewer than 65 bits free, the padding spills into a second block.
